In [ ]:
import os
from pyspark.sql import SparkSession
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Khởi tạo Spark kết nối trực tiếp cụm Standalone và HDFS
spark = SparkSession.builder \
    .appName("FraudDetection_EDA") \
    .master("spark://labsoe-17:7077") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Khởi tạo thành công!")


In [ ]:
# Đường dẫn kho lưu trữ phân tán HDFS mà bạn vừa stream thành công
bronze_path = "hdfs://labsoe-17:9000/lakehouse/fraud_detection/bronze"

print("Đang đọc dữ liệu Delta Table từ HDFS...")
df = spark.read.format("delta").load(bronze_path)

# In schema cấu trúc dữ liệu để báo cáo tài liệu đồ án
df.printSchema()
print(f"Tổng số bản ghi thu thập được trong luồng: {df.count()}")


In [ ]:
from pyspark.sql.functions import col, count, when

print("--- THỐNG KÊ GIÁ TRỊ NULL TRÊN CÁC TRƯỜNG DỮ LIỆU ---")
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).toPandas()
print(null_counts.to_string(index=False))

print("\n--- PHÂN TÍCH HÀM LƯỢNG DỮ LIỆU RÁC (Amount <= 0) ---")
invalid_amount_count = df.filter(col("amount") <= 0).count()
print(f"Số lượng giao dịch có số tiền không hợp lệ (<= 0): {invalid_amount_count} bản ghi")


In [ ]:
print("--- THỐNG KÊ TỶ LỆ MẪU GIAN LẬN (MẤT CÂN BẰNG DỮ LIỆU) ---")
fraud_dist = df.groupBy("is_fraud_label").count().toPandas()
fraud_dist['percentage'] = (fraud_dist['count'] / fraud_dist['count'].sum()) * 100
print(fraud_dist)

# Vẽ biểu đồ trực quan hóa hiện tượng mất cân bằng nhãn (Imbalanced Data)
plt.figure(figsize=(6, 4))
sns.barplot(x="is_fraud_label", y="count", data=fraud_dist, palette="Set2")
plt.title("Phân Phối Nhãn Giao Dịch Gian Lận")
plt.xlabel("Nhãn (0: An toàn, 1: Gian lận)")
plt.ylabel("Số lượng giao dịch")
plt.show()


In [ ]:
print("--- HÀNH VI GIAN LẬN THEO LOẠI HÌNH GIAO DỊCH ---")
type_fraud = df.groupBy("trans_type", "is_fraud_label").count().toPandas()
print(type_fraud)

# Trực quan hóa mối quan hệ giữa Trans_Type và Fraud
plt.figure(figsize=(10, 5))
sns.countplot(x="trans_type", hue="is_fraud_label", data=df.toPandas(), palette="muted")
plt.title("Phân Tích Gian Lận Theo Loại Hình Giao Dịch")
plt.xticks(rotation=45)
plt.show()


In [ ]:
print("--- THỐNG KÊ MÔ TẢ SỐ TIỀN GIAO DỊCH ---")
df.groupBy("is_fraud_label").agg(
    {"amount": "avg", "old_balance_source": "max"}
).show()

# Vẽ biểu đồ Boxplot phân phối số tiền giao dịch để tìm điểm dị biệt (Outliers)
plt.figure(figsize=(8, 5))
sns.boxplot(x="is_fraud_label", y="amount", data=df.sample(False, 0.1).toPandas())
plt.yscale("log") # Dùng thang logarit do độ lệch dòng tiền PaySim rất lớn
plt.title("Biểu Đồ Hộp (Boxplot) Số Tiền Giao Dịch Theo Nhãn")
plt.show()

# Dừng Session sau khi hoàn thành EDA để trả RAM cho máy Master
spark.stop()
